# Pavilion Suitability (S6)

Assesses the single Ryder Street gateway pavilion - the **public, B-KQ-wide information
hub** and civic "living room" where city-core arrivals decide where in B-KQ to go, then
move onward via the **Aston green park** (the onward connector). Aston is one option among
the whole quarter, so the hub **complements, not duplicates** Aston's amenity.

It scores the site on data-backed criteria, then applies a **hard-constraint risk gate**
(helipad safeguarding, land ownership) so a good score never hides a show-stopper. It ends
with a **pavilion vs leverage-existing-space** comparison so the spend is defensible.

Descriptive + appraisal. Site features are OSM (Level 3); ownership/safeguarding are
unverified and flagged for Level 4-5. No funding claim.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Point

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.spatial import audit
from spinelens.models import crossing as cr
from spinelens.models import pavilion as pv

DATA = PHASE1_ROOT / "data"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "pavilion_media"
TABLES = PHASE1_ROOT / "outputs" / "tables"; REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, REPORTS): d.mkdir(parents=True, exist_ok=True)

GATEWAY = (52.484042, -1.892412)
gate_pt = gpd.GeoSeries([Point(GATEWAY[1], GATEWAY[0])], crs=4326).to_crs(27700).iloc[0]

feat = gpd.read_file(DATA / "raw" / "osm_site_features" / "gateway_site_features.geojson").to_crs(27700)
feat["dist_m"] = feat.geometry.distance(gate_pt).round(0)
def label(r):
    for k in ("aeroway", "amenity", "leisure", "landuse", "building"):
        if k in r and pd.notna(r[k]): return f"{k}={r[k]}"
    return "?"
feat["label"] = feat.apply(label, axis=1)

G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected(); largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
cl = {n: c for n, c in coords.items() if n in largest}
nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
def snap(nid): return audit.nearest_node(cl, (float(nodes.loc[nid,"latitude"]), float(nodes.loc[nid,"longitude"])))[0]
print(f"site features: {len(feat)} within 450 m of the gateway")

## 1. Route convergence (data-backed positive)

In [ ]:
CITY_CORE = ["colmore_row", "new_street_station", "moor_street_queensway", "snow_hill_station"]
g_node = snap("ryder_street_pavilion_search_area")
arriving = 0
for o in CITY_CORE:
    try:
        nx.shortest_path(G, snap(o), g_node, weight="length"); arriving += 1
    except nx.NetworkXNoPath:
        pass
print(f"City-core routes converging at the gateway: {arriving}/{len(CITY_CORE)}.")
print("The pavilion sits at the shared-trunk convergence of the city-core spine (notebook 09).")

## 2. Onward access via the Aston green park

In [ ]:
ONWARD = {"aston_university": "Aston University", "millennium_point": "Millennium Point"}
onward_rows = []
for aid, nm in ONWARD.items():
    n = snap(aid)
    netm = nx.shortest_path_length(G, g_node, n, weight="length")
    euc = audit.haversine_m(coords[g_node], coords[n])
    onward_rows.append({"destination": nm, "network_m": round(netm), "straight_m": round(euc)})
onward = pd.DataFrame(onward_rows)
parks = feat[feat["label"].str.contains("park|garden|grass", case=False, na=False)]
print("Onward network distances from the gateway:")
display(onward)
print(f"Green space (park/garden/grass) within 450 m: {len(parks)}; nearest {parks['dist_m'].min():.0f} m.")
print("-> short, green onward access; the Aston park is the connector into the rest of B-KQ.")

## 3. Demand adjacency (students, universities, hospital)

In [ ]:
RES = feat[feat["label"].str.contains("residential|dormitory|apartments", case=False, na=False)]
UNI = feat[feat["label"].str.contains("university|college", case=False, na=False)]
HOSP = feat[feat["label"].str.contains("hospital", case=False, na=False)]
named_res = RES[RES["name"].notna()][["name", "dist_m"]].drop_duplicates().sort_values("dist_m")
print(f"Residential/student buildings within 450 m: {len(RES)} (named: {named_res['name'].nunique()}).")
display(named_res.head(8))
print(f"Universities/colleges: {sorted(UNI['name'].dropna().unique())}")
print(f"Hospital features: {sorted(HOSP['name'].dropna().unique())}")
print("-> strong built-in demand: a cluster of student residences plus Aston and a college.")

## 4. Open space and footprint (constrained)

In [ ]:
gs = gpd.read_file(f"zip://{(DATA/'raw'/'os_open_greenspace'/'opgrsp_essh_sp.zip').as_posix()}!" +
                   [n for n in __import__('zipfile').ZipFile(DATA/'raw'/'os_open_greenspace'/'opgrsp_essh_sp.zip').namelist()
                    if n.lower().replace(' ','').endswith('greenspacesite.shp')][0]).to_crs(27700)
gs_near = int((gs.geometry.distance(gate_pt) <= 250).sum())
grass = feat[feat["label"].str.contains("grass", case=False, na=False)]
print(f"OS Open Greenspace sites within 250 m: {gs_near} (designated greenspace).")
print(f"OSM informal grass within 450 m: {len(grass)}; nearest {grass['dist_m'].min():.0f} m.")
print("-> the site is INFORMAL civic green/public realm, not a designated greenspace: a")
print("   constrained pocket. Usable footprint must be measured on site (FOOTPRINT = medium risk).")

## 5. Helipad safeguarding (HIGH risk, evidenced)

In [ ]:
heli = feat[feat["label"].str.contains("helipad|heliport", case=False, na=False)]
heli_dist = float(heli["dist_m"].min()) if len(heli) else float("nan")
print(f"OSM helipad within 450 m: {len(heli)}; nearest {heli_dist:.0f} m from the gateway.")
print("-> a structure this close to a hospital helipad engages aviation safeguarding:")
print("   height/obstacle limits and CAA + Hospital Trust consent. HELIPAD = HIGH risk.")

## 6. Ownership and consent (HIGH risk, unverified)

In [ ]:
print("Ownership of the site is NOT yet verified. It sits among Aston University land, the")
print("Hospital Trust estate, and Birmingham CC public realm. HM Land Registry INSPIRE parcels")
print("have not been acquired. OWNERSHIP = HIGH risk; the priority validation step is an INSPIRE")
print("parcel screen + Land Registry title check, plus Trust/Aston/Council engagement.")

## 7. Movement / highway conflict

In [ ]:
junction_dist = audit.haversine_m(GATEWAY, (52.4864, -1.8844))
print(f"Distance to the Gate 0C junction (A4540, ~35k veh/day): {junction_dist:.0f} m.")
print("-> proximity gives visibility/footfall (opportunity) but the pavilion must not block")
print("   pedestrian flow at a busy gateway. MOVEMENT = medium risk (design constraint).")

## 8. Suitability scorecard + risk gate

In [ ]:
criteria = {
    "route_convergence": 0.90,   # 4 city-core routes converge here (data-backed)
    "onward_access": 0.90,       # Aston + park adjacent; short green connector
    "demand": 0.95,              # student-residence + university cluster
    "open_space": 0.50,          # informal/constrained civic green, not designated
    "movement": 0.50,            # busy-junction adjacency: opportunity + conflict
    "deliverability": 0.35,      # helipad + ownership constraints
}
weights = {"demand": 0.20, "route_convergence": 0.20, "onward_access": 0.20,
           "open_space": 0.15, "movement": 0.10, "deliverability": 0.15}
score = cr.multi_criteria_rank({"ryder_street_pavilion": criteria}, weights)[0]
risks = {"helipad": "high", "ownership": "high", "footprint": "medium", "movement": "medium"}
gate = pv.risk_gate(risks)
print("Suitability score:", score["score"])
print("Risk gate:", gate)
pd.DataFrame([{"criterion": k, "score": v, "weight": weights[k]} for k, v in criteria.items()]).pipe(display)

## 9. Pavilion vs leverage existing space (defensibility)

In [ ]:
options = {
    "new_pavilion": {"public_info_hub": 1.0, "bkq_wide_orientation": 1.0,
                     "inclusive_public_access": 0.9, "deliverability": 0.40, "cost": 0.40},
    "marker_plus_activation": {"public_info_hub": 0.5, "bkq_wide_orientation": 0.5,
                               "inclusive_public_access": 0.7, "deliverability": 0.80, "cost": 0.85},
}
ow = {"public_info_hub": 0.30, "bkq_wide_orientation": 0.25, "inclusive_public_access": 0.20,
      "deliverability": 0.15, "cost": 0.10}
cmp = pd.DataFrame(cr.multi_criteria_rank(options, ow))
display(cmp)
print("Read: the pavilion wins on the function it exists for (a public, B-KQ-wide information")
print("hub Aston's amenity does not provide). The lighter marker+activation wins on cost/")
print("deliverability and is the fallback if budget or consents tighten.")

## Visuals

In [ ]:
base = ox.graph_to_gdfs(G, nodes=False).to_crs(27700)
minx, miny, maxx, maxy = gate_pt.x-260, gate_pt.y-260, gate_pt.x+260, gate_pt.y+260
cat_color = {"helipad": "#d93025", "hospital": "#ff6d00", "university": "#1a73e8",
             "college": "#1a73e8", "residential": "#9c27b0", "park": "#34a853",
             "garden": "#34a853", "grass": "#a5d6a7"}
def cat(lab):
    for k in cat_color:
        if k in lab: return k
    return None
fig, ax = plt.subplots(figsize=(11, 9))
base.cx[minx:maxx, miny:maxy].plot(ax=ax, color="#e8e8e8", linewidth=0.5, zorder=1)
for _, r in feat.iterrows():
    c = cat(r["label"])
    if c is None: continue
    geom = r.geometry
    pt = geom.centroid
    if not (minx <= pt.x <= maxx and miny <= pt.y <= maxy): continue
    ax.scatter(pt.x, pt.y, s=90 if c in ("helipad","hospital") else 45,
               color=cat_color[c], edgecolor="white", linewidth=0.5, zorder=4)
ax.scatter(gate_pt.x, gate_pt.y, marker="*", s=520, color="black", edgecolor="white", zorder=6)
ax.annotate("pavilion gateway", (gate_pt.x, gate_pt.y), fontsize=9, xytext=(6, 6), textcoords="offset points")
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=v, label=k) for k, v in
           {"helipad": "#d93025", "hospital": "#ff6d00", "university/college": "#1a73e8",
            "student residences": "#9c27b0", "park/green": "#34a853"}.items()]
ax.legend(handles=handles, loc="upper left", fontsize=8)
ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy); ax.set_aspect("equal")
ax.set_title("Pavilion gateway context: helipad, hospital, Aston, student residences, green")
fig.savefig(FIG_DIR / "figN_pavilion_context.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cs = pd.Series(criteria)
axes[0].barh(cs.index, cs.values, color="#1a73e8"); axes[0].set_xlim(0, 1); axes[0].invert_yaxis()
axes[0].set_title(f"Suitability criteria (overall score {score['score']})")
rl = {"helipad": "high", "ownership": "high", "footprint": "medium", "movement": "medium"}
rcol = {"low": "#34a853", "medium": "#fbbc04", "high": "#d93025", "blocker": "#000000"}
axes[1].barh(list(rl), [pv.RISK_LEVELS[v] for v in rl.values()],
             color=[rcol[v] for v in rl.values()])
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(["low", "medium", "high", "blocker"])
axes[1].invert_yaxis(); axes[1].set_title(f"Risk register -> verdict: {gate['verdict']}")
fig.tight_layout(); fig.savefig(FIG_DIR / "figO_suitability_risk.png", dpi=130, bbox_inches="tight"); plt.show()

## Note + ledger

In [ ]:
from spinelens.gate0b import read_csv_rows, write_csv_rows, utc_now_iso
ts = utc_now_iso()
lines = [
    "# Pavilion Suitability Note (S6)",
    "", f"Generated: {ts}. Ryder Street gateway pavilion. Descriptive + appraisal. No funding claim.",
    "", "## Verdict", "",
    f"- Suitability score: **{score['score']}** (criteria below).",
    f"- Risk gate verdict: **{gate['verdict']}** (overall risk {gate['overall_risk']}).",
    f"- Validation required before commitment: {', '.join(gate['validation_items'])}.",
    "", "## Evidence (OSM site features, Level 3)", "",
    f"- Helipad {heli_dist:.0f} m; Aston University ~67 m; Children's Hospital ~89 m.",
    f"- Student/residential buildings within 450 m: {len(RES)} (e.g. {', '.join(named_res['name'].head(3))}).",
    f"- Onward: Aston {int(onward.loc[0,'network_m'])} m, Millennium Point {int(onward.loc[1,'network_m'])} m by network.",
    f"- OS Open Greenspace within 250 m: {gs_near} (designated) - the site is informal civic green.",
    f"- City-core routes converging: {arriving}/{len(CITY_CORE)}.",
    "", "## Scorecard", "",
    "| Criterion | Score |", "|---|---:|",
]
for k, v in criteria.items():
    lines.append(f"| {k} | {v} |")
lines += [
    "", "## Risk register", "",
    "| Risk | Level | Validation step |", "|---|---|---|",
    f"| Helipad safeguarding | high | CAA + Hospital Trust consent; height/obstacle check ({heli_dist:.0f} m to helipad) |",
    "| Land ownership/consent | high | HM Land Registry INSPIRE parcel screen + title; Aston/Trust/Council engagement |",
    "| Usable footprint | medium | On-site survey of the informal green pocket |",
    "| Movement conflict | medium | Pedestrian-flow design so the pavilion does not block the gateway |",
    "", "## Pavilion vs leverage-existing-space", "",
    "The pavilion's unique value is the **public, B-KQ-wide information hub** that Aston's",
    "amphitheatre/lake do not provide. It wins on that function; a lighter marker+activation",
    "wins on cost/deliverability and is the fallback if budget or consents tighten.",
    "", "## Conclusion", "",
    "The Ryder Street site is **suitable in principle and well-justified by demand, convergence",
    "and onward access**, but is **gated by two HIGH risks (helipad safeguarding and land",
    "ownership)** that must be cleared before any funding-facing commitment. Verdict: proceed to",
    "validation, not to build.",
]
(REPORTS / "pavilion_suitability_note.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:20]))

GATE = DATA / "evidence_gate_status_phase1.csv"
g = read_csv_rows(GATE)
g0d = {"gate_id": "G0D", "gate_name": "Pavilion suitability and site risk", "status": "in_progress",
       "owner": "SpineLens", "started_on": "2026-06-04", "completed_on": "",
       "exit_criteria": "Site scored on demand/convergence/onward/open-space/movement; risk register with helipad+ownership; pavilion-vs-leverage comparison",
       "next_action": "Acquire HM Land Registry INSPIRE for ownership screen; CAA/Hospital Trust helipad safeguarding consult; on-site footprint survey",
       "notes": "Verdict: proceed to validation, not build; two HIGH risks gate commitment; no funding claim"}
g = [g0d if r["gate_id"] == "G0D" else r for r in g] if any(r["gate_id"] == "G0D" for r in g) else g + [g0d]
write_csv_rows(GATE, g, list(g[0].keys()))
print("\\ngate G0D recorded.")

## What this unlocks

A defensible, risk-gated pavilion suitability: the site is well-justified by demand,
route convergence and onward access, but commitment is gated by helipad safeguarding and
land ownership - both with explicit validation steps. The pavilion-vs-leverage comparison
keeps the spend honest. Feeds the budget pack (S8) and the validation register (S9).

# Phase 1 Validation Register (S9)

A single, project-wide register of what must be **cross-checked (Level 4)** or
**field-validated (Level 5)** before any funding-facing claim. It consolidates the gaps
surfaced across every gate and experiment (anchors, network, legibility, crossing,
pavilion, corridor, wayfinder, cost, demand) so the proposal is honest about its
evidence frontier.

Evidence ladder: 0 not tested, 1 reachable, 2 raw, 3 audited, 4 cross-checked,
5 field-validated. Funding-facing claims need 4-5.

In [ ]:
from spinelens import validation as val
anchors_df = pd.read_csv(DATA / "study_area_anchors_phase1.csv")
n_anchor = len(anchors_df)
n_unvalidated = int((anchors_df["validation_status"] == "not_validated").sum())

REGISTER = [
    {"item": f"All {n_anchor} Phase 1 anchors are provisional", "domain": "anchors",
     "current_level": 2, "required_level": 5,
     "gap_desc": f"{n_unvalidated}/{n_anchor} anchors not field-validated",
     "validation_action": "Field/site confirmation of exact origin, gateway and crossing points",
     "owner": "SpineLens + field", "priority": "high"},
    {"item": "OSM walk network not cross-checked", "domain": "network",
     "current_level": 3, "required_level": 4, "gap_desc": "Volunteered data vs authoritative roads",
     "validation_action": "Acquire OS Open Roads / OpenMap Local and cross-check",
     "owner": "SpineLens", "priority": "medium"},
    {"item": "Footway width/surface/continuity unverified", "domain": "network",
     "current_level": 3, "required_level": 5, "gap_desc": "Corridor 'severance-free' is from geometry only",
     "validation_action": "On-site footway survey along the corridor", "owner": "field", "priority": "medium"},
    {"item": "RLI weights/thresholds provisional", "domain": "legibility",
     "current_level": 3, "required_level": 4, "gap_desc": "Documented assumptions; landmark/frontage term missing",
     "validation_action": "Extend sensitivity, add POI/frontage term, check literature",
     "owner": "SpineLens", "priority": "medium"},
    {"item": "Perceived (in)legibility not behaviourally verified", "domain": "legibility",
     "current_level": 3, "required_level": 5, "gap_desc": "Model is a structured hypothesis",
     "validation_action": "Walk-through / user audit of low-RLI routes", "owner": "field", "priority": "medium"},
    {"item": "Crossing design unverified (Gate 0C)", "domain": "crossing",
     "current_level": 3, "required_level": 5, "gap_desc": "Warrant evidenced; design not",
     "validation_action": "Speed survey, signal-capacity, swept-path, LTN1/20; BCC/National Highways consent",
     "owner": "BCC / National Highways", "priority": "high"},
    {"item": "Helipad safeguarding (reversible timber pavilion)", "domain": "pavilion",
     "current_level": 2, "required_level": 4, "gap_desc": "Low demountable timber structure ~58 m from helipad; height risk much reduced",
     "validation_action": "Confirm low-height/obstacle clearance with Hospital Trust/CAA for a reversible structure", "owner": "Trust / CAA", "priority": "medium"},
    {"item": "Pavilion site licence (meanwhile-use)", "domain": "pavilion",
     "current_level": 1, "required_level": 4, "gap_desc": "Reversible/meanwhile-use needs a licence, not land acquisition",
     "validation_action": "Meanwhile-use licence with landowner (Aston/Trust/Council); INSPIRE screen to confirm owner",
     "owner": "Aston / Trust / Council", "priority": "medium"},
    {"item": "Pavilion usable footprint unmeasured", "domain": "pavilion",
     "current_level": 2, "required_level": 4, "gap_desc": "Informal green pocket, not designated",
     "validation_action": "On-site footprint survey", "owner": "field", "priority": "medium"},
    {"item": "Corridor segment width/surface are proxies", "domain": "corridor",
     "current_level": 3, "required_level": 4, "gap_desc": "OSM tags, not surveyed",
     "validation_action": "On-site corridor survey", "owner": "field", "priority": "medium"},
    {"item": "Wayfinder effect and unit costs assumed", "domain": "wayfinder",
     "current_level": 2, "required_level": 5, "gap_desc": "Before/after RLI and GBP are assumptions",
     "validation_action": "Footfall/behaviour study + cost validation + accessibility audit",
     "owner": "SpineLens / QS", "priority": "medium"},
    {"item": "All cost figures are placeholders", "domain": "cost",
     "current_level": 1, "required_level": 4, "gap_desc": "Wayfinder/crossing/pavilion GBP indicative",
     "validation_action": "Quantity-surveyor / procurement costing", "owner": "QS", "priority": "high"},
    {"item": "Demand from OSM/AADF only", "domain": "demand",
     "current_level": 3, "required_level": 4, "gap_desc": "No primary footfall/occupancy data",
     "validation_action": "Pedestrian counts + Unite occupancy data", "owner": "SpineLens", "priority": "medium"},
]
for r in REGISTER:
    r.update(val.evidence_gap(r["current_level"], r["required_level"]))
register = pd.DataFrame(REGISTER).sort_values(["priority", "gap"], ascending=[True, False]).reset_index(drop=True)
register.to_csv(TABLES / "phase1_validation_register.csv", index=False)
display(register[["item", "domain", "current_level", "required_level", "gap", "priority"]])

In [ ]:
summary = val.register_summary(REGISTER)
print("Register summary:", summary)
print(f"Open items: {summary['open']}/{summary['items']} | "
      f"require field validation (L5): {summary['field_validation_items']} | "
      f"by priority: {summary['by_priority']}")

In [ ]:
# Evidence-gap chart: current -> required level per item, coloured by priority
reg = register.sort_values(["priority", "gap"], ascending=[True, False])
pcol = {"high": "#d93025", "medium": "#fbbc04", "low": "#34a853"}
fig, ax = plt.subplots(figsize=(11, 7))
y = range(len(reg))
ax.hlines(list(y), reg["current_level"], reg["required_level"],
          color=[pcol[p] for p in reg["priority"]], linewidth=5, alpha=0.7)
ax.scatter(reg["current_level"], list(y), color="#1a73e8", zorder=3, label="current level")
ax.scatter(reg["required_level"], list(y), color="black", marker="|", s=200, zorder=3, label="required level")
ax.set_yticks(list(y)); ax.set_yticklabels(reg["item"], fontsize=8); ax.invert_yaxis()
ax.set_xticks(range(6)); ax.set_xlabel("evidence level (0 not tested -> 5 field validated)")
ax.set_title("Phase 1 validation register: evidence gap to a fundable level")
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color=v, label=k) for k, v in pcol.items()] +
          [mpatches.Patch(color="#1a73e8", label="current"), mpatches.Patch(color="black", label="required")],
          loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "figP_validation_register.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
ts2 = utc_now_iso()
lines = [
    "# Phase 1 Validation Register Note (S9)",
    "", f"Generated: {ts2}. Consolidated Level 4-5 validation needs before funding-facing claims.",
    "",
    f"Open items: {summary['open']}/{summary['items']}. Require field validation (Level 5): "
    f"{summary['field_validation_items']}. By priority: {summary['by_priority']}.",
    "", "## Register (high priority first)", "",
    "| Item | Domain | Current | Required | Validation action | Owner |",
    "|---|---|---:|---:|---|---|",
]
for _, r in register.iterrows():
    lines.append(f"| {r['item']} | {r['domain']} | {r['current_level']} | {r['required_level']} | "
                 f"{r['validation_action']} | {r['owner']} |")
lines += [
    "", "## Reading", "",
    "- Nothing in Phase 1 is yet at a funding-facing evidence level; every headline claim has a",
    "  named validation step and owner.",
    "- The high-priority frontier: provisional anchors, the crossing design, the pavilion helipad",
    "  and ownership, and the cost figures.",
    "- The cheapest high-value next acquisitions are **OS Open Roads** (network cross-check) and",
    "  **HM Land Registry INSPIRE** (pavilion ownership screen).",
]
(REPORTS / "validation_register_note.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:12]))

## What this unlocks

A transparent evidence frontier for the whole of Phase 1: a funder can see exactly what is
proven, what is assumed, and what must be validated next - with owners. This is the
credibility backbone of the proposal and the to-do list that converts the analysis into a
business case.